

# ROC AUC Score

metrics.roc_auc_score(y_true, y_score) gives the value for metrics.auc(x,y) if x=fpr, y=tpr, otherwise metrics.auc(x,y) is a numerical integration for any y(x) over x.


### roc_auc_score(y_true, y_score, average="macro", sample_weight=None):
    =========================================================================
    Compute Area Under the Curve (AUC) from prediction scores
    =========================================================================
    Note: this implementation is restricted to the binary classification task
    or multilabel classification task in label indicator format.
    
    Read more in the :ref:`User Guide <roc_metrics>`.
    
    Parameters
    ----------
    y_true : array, shape = [n_samples] or [n_samples, n_classes]
        True binary labels in binary label indicators.
    
    y_score : array, shape = [n_samples] or [n_samples, n_classes]
       Target scores, can either be probability estimates of the positive
       class, confidence values, or non-thresholded measure of decisions
       (as returned by "decision_function" on some classifiers).
    
    average : string, [None, 'micro', 'macro' (default), 'samples', 'weighted']
       If ``None``, the scores for each class are returned. Otherwise,
       this determines the type of averaging performed on the data:
           ``'micro'``:
           Calculate metrics globally by considering each element of the label
           indicator matrix as a label.
       ``'macro'``:
           Calculate metrics for each label, and find their unweighted
           mean.  This does not take label imbalance into account.
       ``'weighted'``:
           Calculate metrics for each label, and find their average, weighted
           by support (the number of true instances for each label).
       ``'samples'``:
           Calculate metrics for each instance, and find their average.
    
    sample_weight : array-like of shape = [n_samples], optional
       Sample weights.
     
    Returns
    -------
    auc : float
    
    References
    ----------
    .. [1] Wikipedia entry for the Receiver operating characteristic
           https://en.wikipedia.org/wiki/Receiver_operating_characteristic>
    
    =========================================================================

```{python}
    # <...> docstring <...>
    def _binary_roc_auc_score(y_true, y_score, sample_weight=None):
            # <...> bla-bla <...>

            fpr, tpr, tresholds = roc_curve(y_true, y_score,
                                            sample_weight=sample_weight)
            return auc(fpr, tpr, reorder=True)

    return _average_binary_score(
        _binary_roc_auc_score, y_true, y_score, average,
        sample_weight=sample_weight) 
```




# AUC
This will compute the area under \*any\* curve.  If you input x=fpr, y=tpr, then you get the ROC AUC (what we usually refer to as the "AUC").

### auc(x, y, reorder=False)
    =========================================================================
    Compute Area Under the Curve (AUC) using the trapezoidal rule
    =========================================================================
    
    This is a general function, given points on a curve.  For computing the
    area under the ROC-curve, see :func:`roc_auc_score`.
    
    Parameters
    ----------
    x : array, shape = [n]
       x coordinates.
    
    y : array, shape = [n]
       y coordinates.
    
    reorder : boolean, optional (default=False)
       If True, assume that the curve is ascending in the case of ties, as for
       an ROC curve. If the curve is non-ascending, the result will be wrong.
    
    Returns
    -------
    auc : float
    
    Examples
    --------
    >>> import numpy as np
    >>> from sklearn import metrics
    >>> y = np.array([1, 1, 2, 2])
    >>> pred = np.array([0.1, 0.4, 0.35, 0.8])
    >>> fpr, tpr, thresholds = metrics.roc_curve(y, pred, pos_label=2)
    >>> metrics.auc(fpr, tpr)
    0.75
    
   
```{python}

    check_consistent_length(x, y)
    x = column_or_1d(x)
    y = column_or_1d(y)

    if x.shape[0] < 2:
        raise ValueError('At least 2 points are needed to compute
        area under curve, but x.shape = %s' % x.shape)
    
        direction = 1
        if reorder:
            # reorder the data points according to the x axis and using y to
            # break ties
            order = np.lexsort((y, x))
            x, y = x[order], y[order]

        else:
            dx = np.diff(x)
            if np.any(dx < 0):
                if np.all(dx <= 0):
                    direction = -1
                else:
                    raise ValueError("Reordering is not turned on, and "
                                 "the x array is not increasing: %s" % x)

        area = direction * np.trapz(y, x)

        if isinstance(area, np.memmap):
            # Reductions such as .sum used internally in np.trapz do not return a
            # scalar by default for numpy.memmap instances contrary to
            # regular numpy.ndarray instances.
            area = area.dtype.type(area)
    
    return area
```





# roc_curve
### roc_curve(y_true, y_score, pos_label=None, sample_weight=None, drop_intermediate=True)
    
    =========================================================================
    Compute Receiver operating characteristic (ROC)
    =========================================================================
    
    Note: this implementation is restricted to the binary classification task.
    Read more in the :ref:`User Guide <roc_metrics>`.
    
    Parameters
    ----------
    y_true : array, shape = [n_samples]
        True binary labels in range {0, 1} or {-1, 1}.  If labels are not
        binary, pos_label should be explicitly given.

    y_score : array, shape = [n_samples]
        Target scores, can either be probability estimates of the positive
        class, confidence values, or non-thresholded measure of decisions
        (as returned by "decision_function" on some classifiers).

    pos_label : int or str, default=None
        Label considered as positive and others are considered negative.

    sample_weight : array-like of shape = [n_samples], optional
        Sample weights.

    drop_intermediate : boolean, optional (default=True)
        Whether to drop some suboptimal thresholds which would not appear
        on a plotted ROC curve. This is useful in order to create lighter
        ROC curves.

        .. versionadded:: 0.17
           parameter *drop_intermediate*.

    Returns
    -------
    fpr : array, shape = [>2]
        Increasing false positive rates such that element i is the false
        positive rate of predictions with score >= thresholds[i].
        
    tpr : array, shape = [>2]
        Increasing true positive rates such that element i is the true
        positive rate of predictions with score >= thresholds[i].

    thresholds : array, shape = [n_thresholds]
        Decreasing thresholds on the decision function used to compute
        fpr and tpr. `thresholds[0]` represents no instances being predicted
        and is arbitrarily set to `max(y_score) + 1`.

    See also
    --------
    roc_auc_score : Compute Area Under the Curve (AUC) from prediction scores

    Notes
    -----
    Since the thresholds are sorted from low to high values, they
    are reversed upon returning them to ensure they correspond to both ``fpr``
    and ``tpr``, which are sorted in reversed order during their calculation.

    References
    ----------
    .. [1] `Wikipedia entry for the Receiver operating characteristic
            <https://en.wikipedia.org/wiki/Receiver_operating_characteristic>`_
    =======================================================================
    
    
```{python}
def roc_curve(y_true, y_score, pos_label=None, sample_weight=None, drop_intermediate=True)
    
    fps, tps, thresholds = _binary_clf_curve(
        y_true, y_score, pos_label=pos_label, sample_weight=sample_weight)

    # Attempt to drop thresholds corresponding to points in between and
    # collinear with other points. These are always suboptimal and do not
    # appear on a plotted ROC curve (and thus do not affect the AUC).
    # Here np.diff(_, 2) is used as a "second derivative" to tell if there
    # is a corner at the point. Both fps and tps must be tested to handle
    # thresholds with multiple data points (which are combined in
    # _binary_clf_curve). This keeps all cases where the point should be kept,
    # but does not drop more complicated cases like fps = [1, 3, 7],
    # tps = [1, 2, 4]; there is no harm in keeping too many thresholds.

    if drop_intermediate and len(fps) > 2:
        optimal_idxs = np.where(np.r_[True, np.logical_or(np.diff(fps, 2),
                    np.diff(tps, 2)),True])[0]
        fps = fps[optimal_idxs]
        tps = tps[optimal_idxs]
        thresholds = thresholds[optimal_idxs]
        
     if tps.size == 0 or fps[0] != 0:
        # Add an extra threshold position if necessary
        tps = np.r_[0, tps]
        fps = np.r_[0, fps]
        thresholds = np.r_[thresholds[0] + 1, thresholds]

    if fps[-1] <= 0:
        warnings.warn("No negative samples in y_true, "
                      "false positive value should be meaningless",
                      UndefinedMetricWarning)
        fpr = np.repeat(np.nan, fps.shape)
    else:
        fpr = fps / fps[-1]

    if tps[-1] <= 0:
        warnings.warn("No positive samples in y_true, "
                      "true positive value should be meaningless",
                      UndefinedMetricWarning)
        tpr = np.repeat(np.nan, tps.shape)    
    else:
        tpr = tps / tps[-1]

    return fpr, tpr, thresholds
```

_binary_clf_curve(y_true, y_score, pos_label=None, sample_weight=None)
    
    =======================================================================
    Calculate true and false positives per binary classification threshold.
    =======================================================================

    Parameters
    ----------
    y_true : array, shape = [n_samples]
        True targets of binary classification

    y_score : array, shape = [n_samples]
        Estimated probabilities or decision function

    pos_label : int or str, default=None
        The label of the positive class

    sample_weight : array-like of shape = [n_samples], optional
        Sample weights.

    Returns
    -------
    fps : array, shape = [n_thresholds]
        A count of false positives, at index i being the number of negative
        samples assigned a score >= thresholds[i]. The total number of
        negative samples is equal to fps[-1] (thus true negatives are given by
        fps[-1] - fps).

    tps : array, shape = [n_thresholds <= len(np.unique(y_score))]
        An increasing count of true positives, at index i being the number
        of positive samples assigned a score >= thresholds[i]. The total
        number of positive samples is equal to tps[-1] (thus false negatives
        are given by tps[-1] - tps).

    thresholds : array, shape = [n_thresholds]
        Decreasing score values.